# Fraud Detection on Card Transaction Data (Over Sampling)

Machine learning solution for Credit Card Fraud Detection.

The pipeline will include `Decision Tree` and `Random Forest` models, along with `SMOTE` (over-sampling) 
technique on the **positive** (i.e., fraud samples) to deal with imbalanced data points.

⚠️ **Note**

Sampling data with SMOTE may result in longer training time -
~20 mins for each selected model - within a 10x5CV training
schema (default).

This is mainly due to the fact that our experimental pipeline fully leverages
on `GridSearchCV` to implement the CV pipeline, along with model parameter search.
Therefore, while data partitioning is correctly performed
to not incur in any _selection bias_ during the process that
would ultimately inflate the results, it is inevitable that the **same** operations are
applied multiple times (_independently_, ed.) to a given (internal) CV data partition.
In case of over-sampling preprocessing techniques using `SMOTE`, these operations could
be costly.

To avoid this "issue", and make the code more efficient, one should slightly re-invent
the wheel by manually implementing the hyper-param selection of models for each
CV fold selection (therefore, avoiding using `GridSearchCV` entirely).

Alternatively, as a workaround to reduce overall training time, please consider 
reducing the number of repetitions in CV by setting an appropriate value to the `cv_n_reps`
parameter of the `run_experiment` function (e.g. `cv_n_reps=3`).

### Loading Data

In [ ]:
import pandas as pd
import os

project_dir = os.getenv("PROJECT_DIR")
env = os.getenv("CONDA_DEFAULT_ENV")
dataset_csv = os.getenv("DATA")

# Ignore User Warnings
os.environ["PYTHONWARNINGS"] = "ignore"
import warnings

warnings.filterwarnings("ignore")

In [ ]:
print(dataset_csv)

In [ ]:
df = pd.read_csv(dataset_csv)

In [ ]:
df.head()

In [ ]:
df.Class.value_counts()

In [ ]:
X, y = df[df.columns[df.columns != "Class"]], df["Class"]

In [ ]:
X.shape, y.shape

## Experimental Pipeline

In [ ]:
# Reproducibility settings
import numpy as np
from sklearn.utils import check_random_state

SEED = 12345

# The NumPy Generator will be used throughout the whole experiment
# rng = np.random.default_rng(SEED)
np.random.seed(SEED)
rng = check_random_state(SEED)

In [ ]:
# Preprocessing
from sklearn.preprocessing import RobustScaler
from sklearn.compose import ColumnTransformer

# Imbalanced Learning
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import NearMiss

# Model Selection and Metrics
from sklearn.model_selection import train_test_split

# ML Models
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

**Data Splitting**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=rng)

**PreProcessing**

In [ ]:
# (Selected) Feature Scaling
preprocessing = ColumnTransformer(
    [
        ("scaler", RobustScaler(), ["Time", "Amount"]),
    ],
    remainder="passthrough",
)

### Machine Learning Models

Setting up Machine Learning models and their corresponding param grid (for Hyper parameter tuning)

In [ ]:
# Decision Tree
dt = DecisionTreeClassifier(random_state=rng)
tree_models_params = {
    "model__max_depth": [None, 2, 3, 6],
    "model__min_samples_leaf": [2, 5, 6],
    "model__criterion": ["gini", "entropy"],
}

dt_params = tree_models_params

In [ ]:
# Random Forest
rf = RandomForestClassifier(random_state=rng, n_jobs=-1)
rf_params = {
    "model__n_estimators": [
        50,
    ],
    "model__max_features": ["log2", "sqrt"],
}
rf_params_full = tree_models_params | rf_params

#### SMOTE (Over) Sampling Strategy

In [ ]:
smote_run_config = [
    ("Decision Tree", dt, dt_params),
    ("Random Forest", rf, rf_params),  # only RF specific params tuned
]

In [ ]:
# Over Sampling Strategies
smote = SMOTE(sampling_strategy="minority", random_state=rng)

steps_over_sampling = [("preprocess", preprocessing), ("sampling", smote)]

In [ ]:
from fraud_detection.notebook.train import run_experiment

run_experiment(
    name="Over Sampling SMOTE",
    model_configs=smote_run_config,
    data=(X_train, X_test),
    labels=(y_train, y_test),
    preprocessing_steps=steps_over_sampling,
    #    cv_n_reps=3, # uncomment this line, to reduce running time
    rng=rng,
)

---